In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import time

In [2]:
model = load_model(r"D:\deep learning\projects\best_working_dont_touch\final_model.keras")


In [3]:

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [4]:
print(face_cascade.empty())

False


In [5]:
emotion_labels = ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

emotion_colors = {
    "Angry": (0, 0, 255),        # Red
    "Fear": (255, 0, 255),       # Magenta
    "Happy": (0, 255, 0),        # Bright Green
    "Neutral": (255, 165, 0),    # Orange
    "Sad": (255, 100, 0),        # Blue-Orange
    "Surprise": (0, 0, 0)    # black
}

In [6]:
def predict_emotion(face_img):
    face = cv2.resize(face_img, (48, 48))
    face = face.astype("float32") / 255.0
    face = np.expand_dims(face, axis=-1)
    face = np.expand_dims(face, axis=0)

    prediction = model.predict(face, verbose=0)[0]
    idx = np.argmax(prediction)
    confidence = prediction[idx] * 100

    return confidence, prediction

In [7]:
cap=cv2.VideoCapture(0)

In [8]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(80, 80)
    )

    for (x, y, w, h) in faces:
        face_gray = gray[y:y+h, x:x+w]

        confidence, probs = predict_emotion(face_gray)
        idx = np.argmax(probs)
        label = emotion_labels[idx]
        color = emotion_colors[label]

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 3)

        text = f"{label} ({confidence:.1f}%)"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)

        cv2.rectangle(frame, (x, y-35), (x+tw+10, y), color, -1)
        cv2.putText(frame, text, (x+5, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.putText(frame, f"Faces : {len(faces)}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    cv2.imshow("Live Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()